In [45]:
from __future__ import annotations

from typing import TypedDict, List, Annotated, Optional
import operator

from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

import dotenv
import os
dotenv.load_dotenv()

True

In [30]:
class Task(BaseModel):
    id: int
    title: str
    brief: str = Field(..., description="What to cover")


class Plan(BaseModel):
    blog_title: str
    tasks: List[Task]


class State(TypedDict):
    topic: str
    plan: Plan                              # set by orchestrator
    task: Optional[Task]                    # FIX 1: needed so Send("worker", {...}) can inject it
    sections: Annotated[List[str], operator.add]   # reducer: auto-concat worker results
    final: str

In [ ]:
llm = ChatGroq(
    api_key= os.environ.get("GROQ_API_KEY"),
    model=os.environ.get("GROQ_LLM_MODEL")
)

In [32]:
def orchestrator(state: State) -> dict:
    plan = llm.with_structured_output(Plan).invoke(
        [
            SystemMessage(content="Create a blog plan with 5-7 sections on the following topic."),
            HumanMessage(content=f"Topic: {state['topic']}"),
        ]
    )
    return {"plan": plan}

In [33]:
def fanout(state: State):
    # FIX 2: Send keys must be valid State fields — added 'task' to State above
    return [
        Send("worker", {"task": task, "topic": state["topic"], "plan": state["plan"]})
        for task in state["plan"].tasks
    ]

In [34]:
def worker(state: State) -> dict:
    task = state["task"]          # injected by Send
    topic = state["topic"]
    plan = state["plan"]

    section_md = llm.invoke(
        [
            SystemMessage(content="Write one clean Markdown section."),
            HumanMessage(
                content=(
                    f"Blog: {plan.blog_title}\n"
                    f"Topic: {topic}\n\n"
                    f"Section: {task.title}\n"
                    f"Brief: {task.brief}\n\n"
                    "Return only the section content in Markdown."
                )
            ),
        ]
    ).content.strip()

    return {"sections": [section_md]}

In [35]:
from pathlib import Path

def reducer(state: State) -> dict:
    title = state["plan"].blog_title
    body = "\n\n".join(state["sections"]).strip()
    final_md = f"# {title}\n\n{body}\n"

    filename = title.lower().replace(" ", "_") + ".md"
    output_path = Path(filename)
    output_path.write_text(final_md, encoding="utf-8")
    print(f"✅ Blog saved to: {output_path.resolve()}")

    return {"final": final_md}

In [36]:
g = StateGraph(State)
g.add_node("orchestrator", orchestrator)
g.add_node("worker", worker)
g.add_node("reducer", reducer)

g.add_edge(START, "orchestrator")
g.add_conditional_edges("orchestrator", fanout, ["worker"])
g.add_edge("worker", "reducer")
g.add_edge("reducer", END)

app = g.compile()

In [37]:
output = app.invoke({
        "topic": "Write a blog on cricket",
        "sections": [],
    })

✅ Blog saved to: C:\Users\Aayush Bhagat\Desktop\agentic_ai\the_wonderful_world_of_cricket.md
